# Dataset cleanup and merging 

This notebook documents our data cleanup process and merging of the IMIX and Perf Count data.


In [1]:
import pandas as pd
import os
import glob
import numpy as np
import seaborn as sns
import json
import matplotlib.pyplot as plt
import re
import sys
from tqdm import tqdm

sys.path.append('../')
from utils import *
from gatherData import _parse_ncu_report, roofline_results_to_df, calc_roofline_data 

from tqdm.contrib.concurrent import process_map
from functools import partial
import os
from os import path


from sass_helper import *

pd.set_option('display.max_columns', 50)
pd.set_option('display.max_rows', 100)

/Users/gbolet/miniconda3/envs/gpuflopbench-updated/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# let's rename the device column
def rename_devices(x):
    if '3080' in x:
        return '3080'
    elif 'A100' in x:
        return 'A100'
    elif 'A10' in x:
        return 'A10'
    elif 'H100' in x:
        return 'H100'
    else:
        raise ValueError(f'Unknown device name in {x}')

device_to_sm_arch = {
    '3080': 'sm_86',
    'A100': 'sm_80',
    'A10': 'sm_86',
    'H100': 'sm_90'
}   

def fix_omp_kernel_name(x):
    if '__omp_offloading' in x:
        # we need to do a regex here to extract the function name after the following 
        # possible string patterns:
        # __omp_offloading_fd01_2c7d38_
        # __omp_offloading_10305_2b800c7_
        match = re.search(r'__omp_offloading_.*?_.*?_(.+)$', x)
        if match:
            return match.group(1)
    return x

def drop_constant_cols(df):
    df = df.copy()
    nunique = df.nunique()
    cols_to_drop = nunique[nunique == 1].index
    df.drop(cols_to_drop, axis=1, inplace=True)
    return df

# Perf Counter Data Cleanup

In [3]:
# kernel key and metrics
key_cols = ['source', 'Kernel Name', 'exeArgs', 'Block Size', 'Grid Size', 'model_type']
device_col = 'device'

groupings = key_cols + [device_col]
metrics = ['SP_FLOP', 'DP_FLOP', 'HP_FLOP', 'INTOP', 'traffic',
           'bytesRead', 'bytesWrite', 'bytesTotal',
           'dpAI', 'spAI', 'hpAI',
           'dpPerf', 'spPerf', 'hpPerf', 'xtime']

log_metrics = ['traffic', 'xtime']

# all cols
all_cols = groupings + metrics + ['sample']

# markers we should ignore / drop kernels containing these from the dataset
#library_markers = [ 'cub::', 'thrust::', '__cuda_' ]

In [4]:
df = pd.read_csv('all-NCU-GPU-Data.csv', low_memory=False)
print(df.shape)

(20044, 2659)


In [5]:
raw_cols = set(df.columns)

df = drop_constant_cols(df)

after_cols = set(df.columns)

# let's see which columns were dropped
dropped_cols = raw_cols - after_cols
print(f'Dropped columns: {dropped_cols}')


print(df.shape)


Dropped columns: {'lts__t_sectors_srcunit_gcc.avg.peak_sustained', 'sm__ops_path_tensor_src_int8.sum.per_cycle_elapsed', 'device__attribute_gpu_overlap', 'sm__ops_path_tensor_src_int8_sparsity_off.min.per_cycle_elapsed', 'device__attribute_direct_managed_mem_access_from_host', 'device__attribute_handle_type_win32_kmt_handle_supported', 'sm__inst_executed_pipe_fp64_op_dmma.min.pct_of_peak_sustained_elapsed', 'sm__ops_path_tensor_src_tf32_dst_fp32.min.peak_sustained', 'sm__ops_path_tensor_src_tf32_dst_fp32.min.per_second', 'sm__ops_path_tensor_src_tf32_dst_fp32.max.peak_sustained', 'sm__ops_path_tensor_op_igmma_src_int8_sparsity_off.min.per_second', 'sm__ops_path_tensor_op_hgmma_src_fp16_sparsity_off.max', 'sm__ops_path_tensor_src_fp16_dst_fp16_sparsity_on.sum.per_second', 'sm__ops_path_tensor_src_int8.max.peak_sustained', 'sm__ops_path_tensor_op_imma_src_int8_sparsity_on.max.peak_sustained', 'sm__ops_path_tensor_op_imma_src_int8_sparsity_on.sum.per_cycle_elapsed', 'sm__ops_path_tensor_o

In [6]:
df['device'] = df['device'].apply(lambda x: rename_devices(x))
df['SM'] = df['device'].apply(lambda x: device_to_sm_arch[x])
df['Kernel Name'] = df['Kernel Name'].apply(lambda x: fix_omp_kernel_name(x))   

print(df.shape)

(20044, 1364)


In [7]:
# let's write the columns to a file for later use
with open('perf_cntr_cols.txt', 'w') as f:
    for col in df.columns:
        f.write(col + '\n')

In [8]:
display(df[all_cols].head(10))

,source,Kernel Name,exeArgs,Block Size,Grid Size,model_type,device,SP_FLOP,DP_FLOP,HP_FLOP,INTOP,traffic,bytesRead,bytesWrite,bytesTotal,dpAI,spAI,hpAI,dpPerf,spPerf,hpPerf,xtime,sample
0,accuracy-cuda,_Z15accuracy_kerneliiiPKfPKiPi,8192 10000 10 100,"(256, 1, 1)","(2048, 1, 1)",cuda,3080,0,0,0,5.629983e+08,6.449159e+11,327800576,1797888,329598464,0.000000,0.000000,0.0,0.000000e+00,0.000000e+00,0.0,511072.0,1
1,accuracy-cuda,_Z15accuracy_kerneliiiPKfPKiPi,8192 10000 10 100,"(256, 1, 1)","(2048, 1, 1)",cuda,3080,0,0,0,5.629983e+08,6.425681e+11,327824384,1766784,329591168,0.000000,0.000000,0.0,0.000000e+00,0.000000e+00,0.0,512928.0,2
2,accuracy-cuda,_Z15accuracy_kerneliiiPKfPKiPi,8192 10000 10 100,"(256, 1, 1)","(2048, 1, 1)",cuda,3080,0,0,0,5.629983e+08,6.443553e+11,327800320,1800320,329600640,0.000000,0.000000,0.0,0.000000e+00,0.000000e+00,0.0,511520.0,3
3,accuracy-omp,main_l57,8192 10000 10 100,"(128, 1, 1)","(2048, 1, 1)",omp,3080,0,0,0,1.003935e+09,5.279006e+11,327974528,1756416,329730944,0.000000,0.000000,0.0,0.000000e+00,0.000000e+00,0.0,624608.0,1
4,accuracy-omp,main_l57,8192 10000 10 100,"(128, 1, 1)","(2048, 1, 1)",omp,3080,0,0,0,1.003935e+09,5.271992e+11,327968896,1762560,329731456,0.000000,0.000000,0.0,0.000000e+00,0.000000e+00,0.0,625440.0,2
5,accuracy-omp,main_l57,8192 10000 10 100,"(128, 1, 1)","(2048, 1, 1)",omp,3080,0,0,0,1.003935e+09,5.287411e+11,327975808,1772544,329748352,0.000000,0.000000,0.0,0.000000e+00,0.000000e+00,0.0,623648.0,3
6,ace-cuda,_Z14calculateForcePA400_A400_dS1_S1_S1_dddddd,100,"(8, 8, 4)","(50, 50, 100)",cuda,3080,378175731,7817840356,0,7.015186e+09,6.507823e+10,764802432,1538377728,2303180160,3.394368,0.164197,0.0,2.208995e+11,1.068566e+10,0.0,35390944.0,1
7,ace-cuda,_Z9allenCahnPA400_A400_dS1_S1_S1_S1_S1_dddddddd,100,"(8, 8, 4)","(50, 50, 100)",cuda,3080,882719027,14814682409,0,1.476101e+10,6.287344e+10,3050602496,508060544,3558663040,4.162991,0.248048,0.0,2.617416e+11,1.559563e+10,0.0,56600416.0,1
8,ace-cuda,_Z21boundaryConditionsPhiPA400_A400_d,100,"(8, 8, 4)","(50, 50, 100)",cuda,3080,0,0,0,1.084807e+09,5.832238e+10,19200,15426432,15445632,0.000000,0.000000,0.0,0.000000e+00,0.000000e+00,0.0,264832.0,1
9,ace-cuda,_Z15thermalEquationPA400_A400_dS1_S1_S1_ddddd,100,"(8, 8, 4)","(50, 50, 100)",cuda,3080,378358346,7173243211,0,6.858070e+09,8.250915e+10,1779667968,507345664,2287013632,3.136511,0.165438,0.0,2.587908e+11,1.365013e+10,0.0,27718304.0,1


### Let's create some summary counters that are input-size-invariant and thread-count-invariant

In [9]:
ALGO_FAMILY_FEATURES_46 = [

    # base features
    "source", "Kernel Name", "device", "SM", "sample", "exeArgs", "Block Size", "Grid Size", "model_type",

    # Compute / intensity
    "total_AI", "dp_AI", "sp_AI", "hp_AI", "int_AI",
    "dram_bytes_per_thread_inst",

    # Traffic shape
    "dram_read_frac",

    # Exec / structure
    "ipc_active", "issue_to_exec", "smsp_active_cycle_frac",
    "thread_inst_per_thread", "branch_pct", "branch_divergence_pct",

    # Cache / hierarchy
    "l1_to_dram_bytes_ratio", "l2_to_dram_bytes_ratio",
    "l1_hit_pct", "l2_hit_pct",
    "l1_global_ld_hit_pct", "l1_global_st_hit_pct",
    "l2_read_hit_pct", "l2_write_hit_pct",
    "l1_bytes_per_thread_inst", "l2_bytes_per_thread_inst",

    # Coalescing / access signatures
    "gld_bytes_per_sector", "gst_bytes_per_sector",
    "lld_bytes_per_sector", "lst_bytes_per_sector",

    # Shared / atomics / spills
    "shared_bank_conflicts_per_wave", "shared_wavefronts_per_inst",
    "atomic_inst_frac", "spill_inst_frac", "local_spill_pct",

    # Context (mostly size-agnostic; helps avoid false splits)
    "regs_per_thread", "smem_per_block", "theoretical_occupancy_pct",

    # Latency / stall signature
    "avg_warp_latency_per_inst_issued",
    "barrier_stall_ratio", "membar_stall_ratio",
    "long_scoreboard_stall_ratio", "short_scoreboard_stall_ratio",
    "lg_throttle_stall_ratio", "tex_throttle_stall_ratio",
    "math_pipe_throttle_stall_ratio",
    "not_selected_stall_ratio", "imc_miss_stall_ratio", "no_instruction_stall_ratio",
]


def summarize_ncu_algo_family_features_le50(kdf: pd.DataFrame) -> pd.DataFrame:
    """
    Produces a <50-feature subset for clustering kernels by algorithm family.

    Design goals:
      - Favor ratios / normalized metrics (less sensitive to input size).
      - Avoid launch-configuration-specific raw counts (e.g., barrier_count).
      - Keep a small amount of resource/occupancy context to prevent false splits.

    All NCU metric columns assumed to be strings -> apply(str_to_float).
    Missing metrics become NaN.
    """

    out = pd.DataFrame(index=kdf.index)

    out['source'] = kdf['source']
    out['Kernel Name'] = kdf['Kernel Name']
    out['device'] = kdf['device']
    out['SM'] = kdf['SM']
    out['sample'] = kdf['sample']
    out['exeArgs'] = kdf['exeArgs']
    out['Block Size'] = kdf['Block Size']
    out['Grid Size'] = kdf['Grid Size']
    out['model_type'] = kdf['model_type']


    def getf(col: str) -> pd.Series:
        if col in kdf.columns:
            return kdf[col].apply(str_to_float)
        return pd.Series(np.nan, index=kdf.index)

    def safediv(num: pd.Series, den: pd.Series) -> pd.Series:
        den = den.replace(0, np.nan)
        return num / den

    # -------------------------
    # Time / cycles / instructions
    # -------------------------
    time_ns = getf("gpu__time_duration.sum")
    time_s = time_ns * 1e-9

    cycles_per_sec = getf("smsp__cycles_elapsed.avg.per_second")
    cycles_elapsed = getf("smsp__cycles_elapsed.sum")
    cycles_active = getf("smsp__cycles_active.sum")
    out["smsp_active_cycle_frac"] = safediv(cycles_active, cycles_elapsed)

    inst_exec = getf("smsp__inst_executed.sum")
    inst_issued = getf("smsp__inst_issued.sum")
    out["issue_to_exec"] = safediv(inst_issued, inst_exec)

    # IPC active: prefer direct if present
    ipc_direct = getf("sm__inst_executed.avg.per_cycle_active")
    out["ipc_active"] = ipc_direct.where(~ipc_direct.isna(), safediv(inst_exec, cycles_active))

    thread_inst_true = getf("thread_inst_executed_true")
    threads = getf("launch__thread_count")
    out["thread_inst_per_thread"] = safediv(thread_inst_true, threads)

    # -------------------------
    # DRAM traffic + bandwidth + normalized traffic
    # -------------------------
    dram_bytes_read = getf("dram__bytes_read.sum")
    dram_bytes_write = getf("dram__bytes_write.sum")
    dram_bytes_total = dram_bytes_read + dram_bytes_write

    dram_Bps = getf("dram__bytes.sum.per_second")
    dram_Bps = dram_Bps.where(~dram_Bps.isna(), safediv(dram_bytes_total, time_s))

    out["dram_bytes_per_thread_inst"] = safediv(dram_bytes_total, thread_inst_true)
    out["dram_read_frac"] = safediv(dram_bytes_read, dram_bytes_total)

    # -------------------------
    # Ops/sec and arithmetic intensity components
    # -------------------------
    # FP64
    dadd = getf("smsp__sass_thread_inst_executed_op_dadd_pred_on.sum.per_cycle_elapsed")
    dmul = getf("smsp__sass_thread_inst_executed_op_dmul_pred_on.sum.per_cycle_elapsed")
    dfma_x2 = getf("derived__smsp__sass_thread_inst_executed_op_dfma_pred_on_x2")
    if dfma_x2.isna().all():
        dfma = getf("smsp__sass_thread_inst_executed_op_dfma_pred_on.sum.per_cycle_elapsed")
        dfma_x2 = 2.0 * dfma

    # FP32
    fadd = getf("smsp__sass_thread_inst_executed_op_fadd_pred_on.sum.per_cycle_elapsed")
    fmul = getf("smsp__sass_thread_inst_executed_op_fmul_pred_on.sum.per_cycle_elapsed")
    ffma_x2 = getf("derived__smsp__sass_thread_inst_executed_op_ffma_pred_on_x2")
    if ffma_x2.isna().all():
        ffma = getf("smsp__sass_thread_inst_executed_op_ffma_pred_on.sum.per_cycle_elapsed")
        ffma_x2 = 2.0 * ffma

    # FP16 (non-tensor)
    hadd = getf("smsp__sass_thread_inst_executed_op_hadd_pred_on.sum.per_cycle_elapsed")
    hmul = getf("smsp__sass_thread_inst_executed_op_hmul_pred_on.sum.per_cycle_elapsed")
    hfma_x4 = getf("derived__smsp__sass_thread_inst_executed_op_hfma_pred_on_x4")
    if hfma_x4.isna().all():
        hfma = getf("smsp__sass_thread_inst_executed_op_hfma_pred_on.sum.per_cycle_elapsed")
        hfma_x4 = 4.0 * hfma

    dp_ops_s = (dadd + dmul + dfma_x2) * cycles_per_sec
    sp_ops_s = (fadd + fmul + ffma_x2) * cycles_per_sec
    hp_ops_s = ((2.0 * hadd) + (2.0 * hmul) + hfma_x4) * cycles_per_sec

    int_ops = getf("smsp__sass_thread_inst_executed_op_integer_pred_on.sum")
    int_ops_s = safediv(int_ops, time_s)

    out["dp_AI"] = safediv(dp_ops_s, dram_Bps)
    out["sp_AI"] = safediv(sp_ops_s, dram_Bps)
    out["hp_AI"] = safediv(hp_ops_s, dram_Bps)
    out["int_AI"] = safediv(int_ops_s, dram_Bps)

    total_ops_s = dp_ops_s + sp_ops_s + hp_ops_s + int_ops_s
    out["total_AI"] = safediv(total_ops_s, dram_Bps)

    # -------------------------
    # Cache hierarchy fingerprints
    # -------------------------
    l1_r = getf("l1tex__m_xbar2l1tex_read_bytes.sum")
    l1_w = getf("l1tex__m_l1tex2xbar_write_bytes.sum")
    l1_bytes_total = l1_r + l1_w
    l2_bytes_total = getf("derived__lts__lts2xbar_bytes.sum")

    out["l1_to_dram_bytes_ratio"] = safediv(l1_bytes_total, dram_bytes_total)
    out["l2_to_dram_bytes_ratio"] = safediv(l2_bytes_total, dram_bytes_total)

    out["l1_bytes_per_thread_inst"] = safediv(l1_bytes_total, thread_inst_true)
    out["l2_bytes_per_thread_inst"] = safediv(l2_bytes_total, thread_inst_true)

    out["l1_hit_pct"] = getf("l1tex__t_sector_hit_rate.pct")
    out["l2_hit_pct"] = getf("lts__t_sector_hit_rate.pct")
    out["l1_global_ld_hit_pct"] = getf("l1tex__t_sector_pipe_lsu_mem_global_op_ld_hit_rate.pct")
    out["l1_global_st_hit_pct"] = getf("l1tex__t_sector_pipe_lsu_mem_global_op_st_hit_rate.pct")
    out["l2_read_hit_pct"] = getf("lts__t_sector_op_read_hit_rate.pct")
    out["l2_write_hit_pct"] = getf("lts__t_sector_op_write_hit_rate.pct")

    # -------------------------
    # Coalescing / access-width signatures
    # -------------------------
    out["gld_bytes_per_sector"] = getf("smsp__sass_average_data_bytes_per_sector_mem_global_op_ld.ratio")
    out["gst_bytes_per_sector"] = getf("smsp__sass_average_data_bytes_per_sector_mem_global_op_st.ratio")
    out["lld_bytes_per_sector"] = getf("smsp__sass_average_data_bytes_per_sector_mem_local_op_ld.ratio")
    out["lst_bytes_per_sector"] = getf("smsp__sass_average_data_bytes_per_sector_mem_local_op_st.ratio")

    # -------------------------
    # Shared / atomics / spills
    # -------------------------
    sh_waves = getf("l1tex__data_pipe_lsu_wavefronts_mem_shared.sum")
    sh_conf = getf("l1tex__data_bank_conflicts_pipe_lsu_mem_shared.sum")
    out["shared_bank_conflicts_per_wave"] = safediv(sh_conf, sh_waves)
    out["shared_wavefronts_per_inst"] = safediv(sh_waves, inst_exec)

    atom_alu = getf("smsp__inst_executed_op_generic_atom_dot_alu.sum")
    atom_cas = getf("smsp__inst_executed_op_generic_atom_dot_cas.sum")
    atom_sh = getf("smsp__inst_executed_op_shared_atom.sum")
    out["atomic_inst_frac"] = safediv(atom_alu + atom_cas + atom_sh, inst_exec)

    out["local_spill_pct"] = getf("derived__local_spilling_requests_pct")
    spill_inst = getf("sass__inst_executed_register_spilling")
    out["spill_inst_frac"] = safediv(spill_inst, inst_exec)

    # -------------------------
    # Branch + divergence
    # -------------------------
    out["branch_pct"] = getf("derived__smsp__inst_executed_op_branch_pct")
    uniform = getf("smsp__sass_average_branch_targets_threads_uniform.pct")
    out["branch_divergence_pct"] = 100.0 - uniform

    # -------------------------
    # Context: regs/smem/occupancy
    # -------------------------
    out["regs_per_thread"] = getf("launch__registers_per_thread")
    out["smem_per_block"] = getf("launch__shared_mem_per_block")

    occ_parts = [
        getf("launch__occupancy_per_block_size"),
        getf("launch__occupancy_per_register_count"),
        getf("launch__occupancy_per_shared_mem_size"),
        getf("launch__occupancy_per_barrier_count"),
    ]
    out["theoretical_occupancy_pct"] = pd.concat(occ_parts, axis=1).min(axis=1, skipna=True)

    # -------------------------
    # Latency / stalls (ratios are already normalized)
    # -------------------------
    out["avg_warp_latency_per_inst_issued"] = getf("smsp__average_warp_latency_per_inst_executed.ratio")
    # Many reports use ..._per_inst_issued.ratio (your list shows that one), so fallback:
    lat_issued = getf("smsp__average_warp_latency_per_inst_issued.ratio")
    out["avg_warp_latency_per_inst_issued"] = out["avg_warp_latency_per_inst_issued"].where(
        ~out["avg_warp_latency_per_inst_issued"].isna(), lat_issued
    )

    stall_map = {
        "barrier_stall_ratio":         "smsp__average_warps_issue_stalled_barrier_per_issue_active.ratio",
        "membar_stall_ratio":          "smsp__average_warps_issue_stalled_membar_per_issue_active.ratio",
        "long_scoreboard_stall_ratio": "smsp__average_warps_issue_stalled_long_scoreboard_per_issue_active.ratio",
        "short_scoreboard_stall_ratio":"smsp__average_warps_issue_stalled_short_scoreboard_per_issue_active.ratio",
        "lg_throttle_stall_ratio":     "smsp__average_warps_issue_stalled_lg_throttle_per_issue_active.ratio",
        "tex_throttle_stall_ratio":    "smsp__average_warps_issue_stalled_tex_throttle_per_issue_active.ratio",
        "math_pipe_throttle_stall_ratio":"smsp__average_warps_issue_stalled_math_pipe_throttle_per_issue_active.ratio",
        "not_selected_stall_ratio":    "smsp__average_warps_issue_stalled_not_selected_per_issue_active.ratio",
        "imc_miss_stall_ratio":        "smsp__average_warps_issue_stalled_imc_miss_per_issue_active.ratio",
        "no_instruction_stall_ratio":  "smsp__average_warps_issue_stalled_no_instruction_per_issue_active.ratio",
    }
    for out_name, raw_name in stall_map.items():
        out[out_name] = getf(raw_name)

    # Ensure all requested output columns exist (NaN if missing)
    for c in ALGO_FAMILY_FEATURES_46:
        if c not in out.columns:
            out[c] = np.nan

    return out[ALGO_FAMILY_FEATURES_46]


In [10]:
condensed_df = summarize_ncu_algo_family_features_le50(df)
print(condensed_df.shape)
display(condensed_df.head(10))

(20044, 55)


,source,Kernel Name,device,SM,sample,exeArgs,Block Size,Grid Size,model_type,total_AI,dp_AI,sp_AI,hp_AI,int_AI,dram_bytes_per_thread_inst,dram_read_frac,ipc_active,issue_to_exec,smsp_active_cycle_frac,thread_inst_per_thread,branch_pct,branch_divergence_pct,l1_to_dram_bytes_ratio,l2_to_dram_bytes_ratio,l1_hit_pct,...,l1_bytes_per_thread_inst,l2_bytes_per_thread_inst,gld_bytes_per_sector,gst_bytes_per_sector,lld_bytes_per_sector,lst_bytes_per_sector,shared_bank_conflicts_per_wave,shared_wavefronts_per_inst,atomic_inst_frac,spill_inst_frac,local_spill_pct,regs_per_thread,smem_per_block,theoretical_occupancy_pct,avg_warp_latency_per_inst_issued,barrier_stall_ratio,membar_stall_ratio,long_scoreboard_stall_ratio,short_scoreboard_stall_ratio,lg_throttle_stall_ratio,tex_throttle_stall_ratio,math_pipe_throttle_stall_ratio,not_selected_stall_ratio,imc_miss_stall_ratio,no_instruction_stall_ratio
0,accuracy-cuda,_Z15accuracy_kerneliiiPKfPKiPi,3080,sm_86,1,8192 10000 10 100,"(256, 1, 1)","(2048, 1, 1)",cuda,1.708134,0.000000,0.000000,0.0,1.708134,0.356977,0.994545,0.66,1.000449,0.931732,1761.064590,0.10,0.59,0.995487,NaN,1.23,...,0.355366,NaN,31.65,0.00,0.0,0.0,0.006783,0.006224,6.707808e-05,0.0,0.0,24.0,1068.0,1992.0,71.02,3.98,0.0,61.47,0.11,0.0,0.00,0.21,0.24,0.06,0.21
1,accuracy-cuda,_Z15accuracy_kerneliiiPKfPKiPi,3080,sm_86,2,8192 10000 10 100,"(256, 1, 1)","(2048, 1, 1)",cuda,1.708172,0.000000,0.000000,0.0,1.708172,0.356969,0.994639,0.66,1.000448,0.931513,1761.064590,0.10,0.59,0.995496,NaN,1.23,...,0.355361,NaN,31.65,0.00,0.0,0.0,0.006971,0.006225,6.707808e-05,0.0,0.0,24.0,1068.0,1992.0,71.06,3.94,0.0,61.51,0.11,0.0,0.00,0.21,0.24,0.06,0.21
2,accuracy-cuda,_Z15accuracy_kerneliiiPKfPKiPi,3080,sm_86,3,8192 10000 10 100,"(256, 1, 1)","(2048, 1, 1)",cuda,1.708123,0.000000,0.000000,0.0,1.708123,0.356979,0.994538,0.66,1.000447,0.929327,1761.064590,0.10,0.59,0.995478,NaN,1.23,...,0.355365,NaN,31.65,0.00,0.0,0.0,0.006713,0.006226,6.707808e-05,0.0,0.0,24.0,1068.0,1992.0,71.05,3.96,0.0,61.49,0.11,0.0,0.00,0.21,0.24,0.06,0.21
3,accuracy-omp,main_l57,3080,sm_86,1,8192 10000 10 100,"(128, 1, 1)","(2048, 1, 1)",omp,3.044709,0.000000,0.000000,0.0,3.044709,0.181821,0.994673,1.04,1.000163,0.956368,6917.931343,0.14,0.28,0.994852,NaN,0.31,...,0.180885,NaN,31.89,0.00,0.0,0.0,0.002014,0.015852,6.505204e-08,0.0,0.0,56.0,1220.0,1052.0,30.61,1.40,0.0,24.02,0.13,0.0,0.00,0.24,0.29,0.02,0.12
4,accuracy-omp,main_l57,3080,sm_86,2,8192 10000 10 100,"(128, 1, 1)","(2048, 1, 1)",omp,3.044704,0.000000,0.000000,0.0,3.044704,0.181821,0.994655,1.04,1.000162,0.948161,6917.931343,0.14,0.28,0.994855,NaN,0.30,...,0.180886,NaN,31.89,0.00,0.0,0.0,0.001795,0.015850,6.505204e-08,0.0,0.0,56.0,1220.0,1052.0,30.69,1.42,0.0,23.98,0.13,0.0,0.00,0.25,0.29,0.02,0.12
5,accuracy-omp,main_l57,3080,sm_86,3,8192 10000 10 100,"(128, 1, 1)","(2048, 1, 1)",omp,3.044548,0.000000,0.000000,0.0,3.044548,0.181830,0.994625,1.05,1.000162,0.953835,6917.931343,0.14,0.28,0.994809,NaN,0.30,...,0.180886,NaN,31.89,0.00,0.0,0.0,0.001898,0.015852,6.505204e-08,0.0,0.0,56.0,1220.0,1052.0,30.62,1.40,0.0,24.00,0.13,0.0,0.00,0.24,0.29,0.02,0.12
6,ace-cuda,_Z14calculateForcePA400_A400_dS1_S1_S1_dddddd,3080,sm_86,1,100,"(8, 8, 4)","(50, 50, 100)",cuda,6.604434,3.394368,0.164197,0.0,3.045870,0.111559,0.332064,0.21,1.000024,0.999751,322.583571,0.11,0.00,1.313294,NaN,41.19,...,0.146510,NaN,27.37,31.76,0.0,0.0,0.000000,0.016298,0.000000e+00,0.0,0.0,52.0,1024.0,936.0,141.33,0.00,0.0,0.00,67.98,0.0,67.82,0.02,0.89,0.00,0.24
7,ace-cuda,_Z9allenCahnPA400_A400_dS1_S1_S1_S1_S1_dddddddd,3080,sm_86,1,100,"(8, 8, 4)","(50, 50, 100)",cuda,8.558948,4.162991,0.248048,0.0,4.147909,0.095956,0.857233,0.23,1.000014,0.999760,579.475819,0.10,0.00,1.427071,NaN,40.92,...,0.136936,NaN,27.93,31.84,0.0,0.0,0.000000,0.004688,0.000000e+00,0.0,0.0,42.0,1024.0,968.0,163.60,0.00,0.0,0.58,72.12,0.0,84.70,0.01,1.20,0.00,0.16
8,ace-cuda,_Z21boundaryConditionsPhiPA400_A400_d,3080,sm_86,1,100,"(8, 8, 4)","(50, 50, 100)",cuda,70.233914,0.000000,0.00

In [11]:
# let's average out the samples for each kernel
avrgd_df = condensed_df.groupby(['source', 'Kernel Name', 'device', 'SM', 'exeArgs', 'Block Size', 'Grid Size', 'model_type']).mean().reset_index()

# drop the "sample" column since it's no longer relevant after averaging
avrgd_df = avrgd_df.drop(columns=['sample'])

print(avrgd_df.shape)
display(avrgd_df.head(20))


(5362, 54)


,source,Kernel Name,device,SM,exeArgs,Block Size,Grid Size,model_type,total_AI,dp_AI,sp_AI,hp_AI,int_AI,dram_bytes_per_thread_inst,dram_read_frac,ipc_active,issue_to_exec,smsp_active_cycle_frac,thread_inst_per_thread,branch_pct,branch_divergence_pct,l1_to_dram_bytes_ratio,l2_to_dram_bytes_ratio,l1_hit_pct,l2_hit_pct,...,l1_bytes_per_thread_inst,l2_bytes_per_thread_inst,gld_bytes_per_sector,gst_bytes_per_sector,lld_bytes_per_sector,lst_bytes_per_sector,shared_bank_conflicts_per_wave,shared_wavefronts_per_inst,atomic_inst_frac,spill_inst_frac,local_spill_pct,regs_per_thread,smem_per_block,theoretical_occupancy_pct,avg_warp_latency_per_inst_issued,barrier_stall_ratio,membar_stall_ratio,long_scoreboard_stall_ratio,short_scoreboard_stall_ratio,lg_throttle_stall_ratio,tex_throttle_stall_ratio,math_pipe_throttle_stall_ratio,not_selected_stall_ratio,imc_miss_stall_ratio,no_instruction_stall_ratio
0,accuracy-cuda,_Z15accuracy_kerneliiiPKfPKiPi,3080,sm_86,8192 10000 10 100,"(256, 1, 1)","(2048, 1, 1)",cuda,1.708143,0.000000,0.000000,0.000000,1.708143,0.356975,0.994574,0.660000,1.000448,0.930857,1761.064590,0.10,0.59,0.995487,NaN,1.230000,0.150000,...,0.355364,NaN,31.65,0.00,0.0,0.0,0.006822,0.006225,6.707808e-05,0.0,0.0,24.0,1068.0,1992.0,71.043333,3.960000,0.0,61.490000,0.110000,0.0,0.000000,0.210000,0.240000,0.06,0.210000
1,accuracy-cuda,_Z15accuracy_kerneliiiPKfPKiPi,A10,sm_86,8192 10000 10 100,"(256, 1, 1)","(2048, 1, 1)",cuda,1.428051,0.000000,0.000000,0.000000,1.428051,0.426990,0.994508,0.696667,1.000471,0.981820,1761.064590,0.10,0.59,0.832251,NaN,1.240000,0.173333,...,0.355363,NaN,31.65,0.00,0.0,0.0,0.007349,0.006232,6.707808e-05,0.0,0.0,24.0,1068.0,1992.0,65.823333,2.920000,0.0,57.300000,0.110000,0.0,0.000000,0.220000,0.250000,0.06,0.210000
2,accuracy-cuda,_Z15accuracy_kerneliiiPKfPKiPi,A100,sm_80,8192 10000 10 100,"(256, 1, 1)","(2048, 1, 1)",cuda,1.732228,0.000000,0.000000,0.025408,1.706820,0.357381,0.992787,0.710000,1.000771,0.966820,1762.064590,0.10,0.59,0.993726,NaN,1.290000,23.700000,...,0.355139,NaN,31.65,0.00,0.0,0.0,0.006525,0.006229,6.704210e-05,0.0,0.0,24.0,1068.0,3536.0,74.156667,2.000000,0.0,64.346667,0.150000,0.0,0.000000,0.963333,1.176667,0.12,0.280000
3,accuracy-cuda,_Z15accuracy_kerneliiiPKfPKiPi,H100,sm_90,8192 10000 10 100,"(256, 1, 1)","(2048, 1, 1)",cuda,1.474491,0.000000,0.000000,0.025406,1.449085,0.384969,0.992768,0.853333,1.000842,0.962453,1635.888809,0.10,0.59,0.993695,NaN,1.180000,0.653333,...,0.382541,NaN,31.65,0.00,0.0,0.0,0.014775,0.006688,7.182360e-05,0.0,0.0,18.0,1068.0,1440.0,71.513333,3.430000,0.0,61.853333,0.230000,0.0,0.000000,0.120000,0.233333,0.03,0.246667
4,accuracy-omp,main_l57,3080,sm_86,8192 10000 10 100,"(128, 1, 1)","(2048, 1, 1)",omp,3.044653,0.000000,0.000000,0.000000,3.044653,0.181824,0.994651,1.043333,1.000162,0.952788,6917.931343,0.14,0.28,0.994839,NaN,0.303333,0.120000,...,0.180886,NaN,31.89,0.00,0.0,0.0,0.001902,0.015851,6.505204e-08,0.0,0.0,56.0,1220.0,1052.0,30.640000,1.406667,0.0,24.000000,0.130000,0.0,0.000000,0.243333,0.290000,0.02,0.120000
5,accuracy-omp,main_l57,A10,sm_86,8192 10000 10 100,"(128, 1, 1)","(2048, 1, 1)",omp,2.400447,0.000000,0.000000,0.000000,2.400447,0.230620,0.994717,1.186667,1.000172,0.898966,6917.931343,0.14,0.28,0.784312,NaN,0.310000,0.130000,...,0.180878,NaN,31.89,0.00,0.0,0.0,0.001963,0.015852,6.505204e-08,0.0,0.0,56.0,1220.0,1052.0,28.356667,1.186667,0.0,21.863333,0.130000,0.0,0.000000,0.280000,0.320000,0.02,0.120000
6,accuracy-omp,main_l57,A100,sm_80,8192 10000 10 100,"(128, 1, 1)","(2048, 1, 1)",omp,3.052523,0.000000,0.000000,0.012703,3.039820,0.182113,0.992831,0.900000,1.000258,0.877131,6917.931343,0.14,0.28,0.993086,NaN,0.340000,23.716667,...,0.180854,NaN,31.89,0.00,0.0,0.0,0.001507,0.015846,6.505204e-08,0.0,0.0,56.0,1220.0,1800.0,33.570000,0.856667,0.0,26.803333,0.120000,0.0,0.000000,0.520000,0.580000,0.03,0.140000
7,accuracy-omp,main_l57,H100,sm_90,8192 10000 10 100,"(128, 1, 1)","(2048, 1, 1)",omp,2.822523,0.000000,0.000000,0.000122,2.822401,0

# IMIX Data Cleanup

In [12]:
sass = pd.read_csv('imix_data.csv', low_memory=False)
print(sass.shape)
print(sass.columns)

display(sass.head(10))

(14757, 151)
Index(['Kernel Name', 'Program', 'Model', 'SM', 'Num Labels', 'Num References',
       'Num Self References', 'Num Circular Dependencies', 'Num Predicated',
       'Num Lines',
       ...
       'TEX', 'BRXU', 'I2FP', 'F2IP', 'VIADD', 'VIADDMNMX', 'ENDCOLLECTIVE',
       'VIMNMX', 'CGAERRBAR', 'VIMNMX3'],
      dtype='object', length=151)


,Kernel Name,Program,Model,SM,Num Labels,Num References,Num Self References,Num Circular Dependencies,Num Predicated,Num Lines,Num Constant Math,Num FP16,Num FP32,Num FP64,Num Global Loads,Num Global Stores,Labels,References,OpType_movement,OpType_integer,OpType_synchronization,OpType_control,OpType_floating point,MOV,LOP3,...,UP2UR,UPRMT,USGXT,NANOSLEEP,BREV,UPLOP3,B2R,BMSK,HMMA,HSET2,VABSDIFF4,UBREV,MATCH,OpType_texture,AddrSpace_texture,TEX,BRXU,I2FP,F2IP,VIADD,VIADDMNMX,ENDCOLLECTIVE,VIMNMX,CGAERRBAR,VIMNMX3
0,__cuda_sm20_sqrt_rn_f32_slowpath,vmc,cuda,sm_80,4,0,0,0,15,32,5,0,12,0,0,0,".L_x_10,.L_x_9,.L_x_11,.L_x_59",NaN,4.0,1.0,2.0,13,12.0,4.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,_Z9propagateiiPfS_S_S_S_S_S_S_Pj,vmc,cuda,sm_80,33,6,0,0,9,264,112,4,86,0,12,12,".L_x_36,.L_x_0,.L_x_15,.L_x_16,.L_x_14,.L_x_1,...","__cuda_sm20_sqrt_rn_f32_slowpath,__cuda_sm20_s...",25.0,59.0,16.0,40,90.0,49.0,13.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,_Z10initializePfS_S_S_S_S_S_Pj,vmc,cuda,sm_80,14,3,0,0,3,136,70,4,50,0,1,8,".L_x_6,.L_x_39,.L_x_40,.L_x_38,.L_x_7,.L_x_42,...","__cuda_sm20_sqrt_rn_f32_slowpath,__cuda_sm20_s...",11.0,28.0,6.0,19,54.0,23.0,9.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,_Z10zero_statsiPf,vmc,cuda,sm_80,2,0,0,0,0,32,8,1,0,0,0,4,".L_x_48,.L_x_62",NaN,2.0,7.0,0.0,15,1.0,2.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,_Z7initranjPj,vmc,cuda,sm_80,2,0,0,0,0,24,6,1,0,0,0,1,".L_x_49,.L_x_63",NaN,2.0,5.0,0.0,12,1.0,2.0,2.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5,_Z15SumWithinBlocksiPKfPf,vmc,cuda,sm_80,8,0,0,0,42,152,65,4,14,0,5,1,".L_x_54,.L_x_53,.L_x_52,.L_x_55,.L_x_51,.L_x_5...",NaN,3.0,64.0,13.0,17,18.0,3.0,3.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
6,_Z15LCG_random_initPj,vmc,cuda,sm_80,2,0,0,0,0,16,3,1,0,0,0,0,".L_x_57,.L_x_65",NaN,0.0,2.0,0.0,10,1.0,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
7,_Z10LCG_randomPj,vmc,cuda,sm_80,2,0,0,0,0,24,4,1,1,0,0,0,".L_x_58,.L_x_66",NaN,2.0,2.0,0.0,14,2.0,2.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
8,__kmpc_get_hardware_thread_id_in_block,sort,omp,sm_80,2,0,0,0,0,16,0,0,0,0,0,0,".L_x_4,.L_x_40",NaN,0.0,0.0,0.0,15,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
9,__omp_offloading_fd01_183cbd_main_l146,sort,omp,sm_80,16,1,0,0,35,304,118,2,1,0,5,4,".L_x_0,.L_x_17,.L_x_7,.L_x_6,.L_x_12,.L_x_9,.L...",__kmpc_get_hardware_thread_id_in_block,8.0,122.0,27.0,28,3.0,4.0,6.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [13]:
# need to rename some of the columns and make a few new ones to match the NCU data
# some of the rows in the imix data will not appear in the perf counter data, as some functions are __device__ functions that are inlined or called by the __global__ kernels

sass['source'] = sass['Program'] + '-' + sass['Model']

In [14]:


sass['Kernel Name'] = sass['Kernel Name'].apply(lambda x: fix_omp_kernel_name(x))

# Merge the Dataframes

In [15]:
# let's match up the 'source', 'Kernel Name', and 'SM' columns in the perf counter data with the 'source', 'Kernel Name', and 'SM' columns in the imix data, and add the imix metrics to the perf counter dataframe

df_merged = pd.merge(avrgd_df, sass, on=['source', 'Kernel Name', 'SM'], how='left', suffixes=('', '_sass_overlap'))

In [16]:
assert not df_merged.columns.str.contains('sass_overlap').any(), "There exist sass_overlap columns in the merged dataframe, which means there were some columns in the imix data that had the same name as columns in the perf counter data. This should not happen, since we added the _sass_overlap suffix to any overlapping columns. Please check the column names in both dataframes and resolve any overlaps before merging."

In [17]:
print(df_merged.shape)
print(df_merged.columns)
print(len(df_merged.columns))

(5398, 203)
Index(['source', 'Kernel Name', 'device', 'SM', 'exeArgs', 'Block Size',
       'Grid Size', 'model_type', 'total_AI', 'dp_AI',
       ...
       'TEX', 'BRXU', 'I2FP', 'F2IP', 'VIADD', 'VIADDMNMX', 'ENDCOLLECTIVE',
       'VIMNMX', 'CGAERRBAR', 'VIMNMX3'],
      dtype='object', length=203)
203


In [18]:
# check if there are any NaN columns in the merged dataframe, which would indicate that there were some columns in the imix data that did not have a match in the perf counter data
nan_cols = df_merged.columns[df_merged.isna().all()].tolist()
print(f'Columns with NaN values after merge: {nan_cols}')

Columns with NaN values after merge: ['l2_to_dram_bytes_ratio', 'l2_bytes_per_thread_inst']


In [19]:

display(df_merged.head(15))

,source,Kernel Name,device,SM,exeArgs,Block Size,Grid Size,model_type,total_AI,dp_AI,sp_AI,hp_AI,int_AI,dram_bytes_per_thread_inst,dram_read_frac,ipc_active,issue_to_exec,smsp_active_cycle_frac,thread_inst_per_thread,branch_pct,branch_divergence_pct,l1_to_dram_bytes_ratio,l2_to_dram_bytes_ratio,l1_hit_pct,l2_hit_pct,...,UP2UR,UPRMT,USGXT,NANOSLEEP,BREV,UPLOP3,B2R,BMSK,HMMA,HSET2,VABSDIFF4,UBREV,MATCH,OpType_texture,AddrSpace_texture,TEX,BRXU,I2FP,F2IP,VIADD,VIADDMNMX,ENDCOLLECTIVE,VIMNMX,CGAERRBAR,VIMNMX3
0,accuracy-cuda,_Z15accuracy_kerneliiiPKfPKiPi,3080,sm_86,8192 10000 10 100,"(256, 1, 1)","(2048, 1, 1)",cuda,1.708143,0.000000,0.000000,0.000000,1.708143,0.356975,0.994574,0.660000,1.000448,0.930857,1761.064590,0.10,0.59,0.995487,NaN,1.230000,0.150000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,accuracy-cuda,_Z15accuracy_kerneliiiPKfPKiPi,A10,sm_86,8192 10000 10 100,"(256, 1, 1)","(2048, 1, 1)",cuda,1.428051,0.000000,0.000000,0.000000,1.428051,0.426990,0.994508,0.696667,1.000471,0.981820,1761.064590,0.10,0.59,0.832251,NaN,1.240000,0.173333,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,accuracy-cuda,_Z15accuracy_kerneliiiPKfPKiPi,A100,sm_80,8192 10000 10 100,"(256, 1, 1)","(2048, 1, 1)",cuda,1.732228,0.000000,0.000000,0.025408,1.706820,0.357381,0.992787,0.710000,1.000771,0.966820,1762.064590,0.10,0.59,0.993726,NaN,1.290000,23.700000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,accuracy-cuda,_Z15accuracy_kerneliiiPKfPKiPi,H100,sm_90,8192 10000 10 100,"(256, 1, 1)","(2048, 1, 1)",cuda,1.474491,0.000000,0.000000,0.025406,1.449085,0.384969,0.992768,0.853333,1.000842,0.962453,1635.888809,0.10,0.59,0.993695,NaN,1.180000,0.653333,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,3.0,0.0,0.0,0.0,0.0,0.0
4,accuracy-omp,main_l57,3080,sm_86,8192 10000 10 100,"(128, 1, 1)","(2048, 1, 1)",omp,3.044653,0.000000,0.000000,0.000000,3.044653,0.181824,0.994651,1.043333,1.000162,0.952788,6917.931343,0.14,0.28,0.994839,NaN,0.303333,0.120000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5,accuracy-omp,main_l57,A10,sm_86,8192 10000 10 100,"(128, 1, 1)","(2048, 1, 1)",omp,2.400447,0.000000,0.000000,0.000000,2.400447,0.230620,0.994717,1.186667,1.000172,0.898966,6917.931343,0.14,0.28,0.784312,NaN,0.310000,0.130000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
6,accuracy-omp,main_l57,A100,sm_80,8192 10000 10 100,"(128, 1, 1)","(2048, 1, 1)",omp,3.052523,0.000000,0.000000,0.012703,3.039820,0.182113,0.992831,0.900000,1.000258,0.877131,6917.931343,0.14,0.28,0.993086,NaN,0.340000,23.716667,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
7,accuracy-omp,main_l57,H100,sm_90,8192 10000 10 100,"(128, 1, 1)","(2048, 1, 1)",omp,2.822523,0.000000,0.000000,0.000122,2.822401,0.186997,0.992694,1.026667,1.000068,0.939985,6739.118828,0.14,0.32,0.992776,NaN,0.310000,0.843333,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,21.0,2.0,14.0,0.0,0.0,0.0
8,ace-cuda,_Z14calculateForcePA400_A400_dS1_S1_S1_dddddd,3080,sm_86,100,"(8, 8, 4)","(50, 50, 100)",cuda,6.604762,3.394530,0.164198,0.000000,3.046033,0.111553,0.332041,0.210000,1.000024,0.999811,322.583571,0.11,0.00,1.312885,NaN,41.196667,74.733333,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
9,ace-cuda,_Z14calculateForcePA400_A400_dS1_S1_S1_dddddd,A10,sm_86,100,"(8, 8, 4)","(50, 50, 100)",cuda,5.114131,2.628421,0.127142,0.000000,2.358568,0.144068,0.359107,0.210000,1.000025,0.999673,322.583571,0.11,0.00,1.015367,NaN,41.310000,74.633333,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [20]:

subset = sass[(sass['Kernel Name'] == '_Z12geglu_kernelIffLi160ELi1280ELi8ELi2EEvPT_PKS0_') & 
              (sass['SM'] == 'sm_90')]

display(subset)


,Kernel Name,Program,Model,SM,Num Labels,Num References,Num Self References,Num Circular Dependencies,Num Predicated,Num Lines,Num Constant Math,Num FP16,Num FP32,Num FP64,Num Global Loads,Num Global Stores,Labels,References,OpType_movement,OpType_integer,OpType_synchronization,OpType_control,OpType_floating point,MOV,LOP3,...,UPRMT,USGXT,NANOSLEEP,BREV,UPLOP3,B2R,BMSK,HMMA,HSET2,VABSDIFF4,UBREV,MATCH,OpType_texture,AddrSpace_texture,TEX,BRXU,I2FP,F2IP,VIADD,VIADDMNMX,ENDCOLLECTIVE,VIMNMX,CGAERRBAR,VIMNMX3,source
10901,_Z12geglu_kernelIffLi160ELi1280ELi8ELi2EEvPT_P...,geglu,cuda,sm_90,2,0,0,0,48,472,407,6,400,0,8,4,".L_x_2,.L_x_8",NaN,5.0,26.0,0.0,17,406.0,5.0,16.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,geglu-cuda


In [21]:

subset = sass[(sass['source'] == 'geglu-cuda') & 
              (sass['SM'] == 'sm_90')]

display(subset)

,Kernel Name,Program,Model,SM,Num Labels,Num References,Num Self References,Num Circular Dependencies,Num Predicated,Num Lines,Num Constant Math,Num FP16,Num FP32,Num FP64,Num Global Loads,Num Global Stores,Labels,References,OpType_movement,OpType_integer,OpType_synchronization,OpType_control,OpType_floating point,MOV,LOP3,...,UPRMT,USGXT,NANOSLEEP,BREV,UPLOP3,B2R,BMSK,HMMA,HSET2,VABSDIFF4,UBREV,MATCH,OpType_texture,AddrSpace_texture,TEX,BRXU,I2FP,F2IP,VIADD,VIADDMNMX,ENDCOLLECTIVE,VIMNMX,CGAERRBAR,VIMNMX3,source
10899,_Z12geglu_kernelIffLi160ELi5120ELi8ELi2EEvPT_P...,geglu,cuda,sm_90,2,0,0,0,48,480,436,4,400,0,8,4,".L_x_0,.L_x_6",NaN,3.0,40.0,0.0,15,404.0,3.0,19.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,geglu-cuda
10900,_Z12geglu_kernelIffLi160ELi2560ELi8ELi2EEvPT_P...,geglu,cuda,sm_90,2,0,0,0,48,472,432,4,400,0,8,4,".L_x_1,.L_x_7",NaN,3.0,37.0,0.0,10,404.0,3.0,18.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,geglu-cuda
10901,_Z12geglu_kernelIffLi160ELi1280ELi8ELi2EEvPT_P...,geglu,cuda,sm_90,2,0,0,0,48,472,407,6,400,0,8,4,".L_x_2,.L_x_8",NaN,5.0,26.0,0.0,17,406.0,5.0,16.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,geglu-cuda
10902,_Z12geglu_kernelIffLi160ELi5120ELi8ELi1EEvPT_P...,geglu,cuda,sm_90,2,0,0,0,24,256,211,5,200,0,4,2,".L_x_3,.L_x_9",NaN,4.0,20.0,0.0,15,205.0,4.0,9.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,geglu-cuda
10903,_Z12geglu_kernelIffLi160ELi2560ELi8ELi1EEvPT_P...,geglu,cuda,sm_90,2,0,0,0,24,256,211,5,200,0,4,2,".L_x_4,.L_x_10",NaN,4.0,20.0,0.0,15,205.0,4.0,10.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,geglu-cuda
10904,_Z12geglu_kernelIffLi160ELi1280ELi8ELi1EEvPT_P...,geglu,cuda,sm_90,2,0,0,0,24,248,208,5,200,0,4,2,".L_x_5,.L_x_11",NaN,4.0,17.0,0.0,10,205.0,4.0,8.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,geglu-cuda


In [22]:
print(df_merged.shape)

print(df_merged.head(10))


(5398, 203)
          source                                    Kernel Name device     SM  \
0  accuracy-cuda                 _Z15accuracy_kerneliiiPKfPKiPi   3080  sm_86   
1  accuracy-cuda                 _Z15accuracy_kerneliiiPKfPKiPi    A10  sm_86   
2  accuracy-cuda                 _Z15accuracy_kerneliiiPKfPKiPi   A100  sm_80   
3  accuracy-cuda                 _Z15accuracy_kerneliiiPKfPKiPi   H100  sm_90   
4   accuracy-omp                                       main_l57   3080  sm_86   
5   accuracy-omp                                       main_l57    A10  sm_86   
6   accuracy-omp                                       main_l57   A100  sm_80   
7   accuracy-omp                                       main_l57   H100  sm_90   
8       ace-cuda  _Z14calculateForcePA400_A400_dS1_S1_S1_dddddd   3080  sm_86   
9       ace-cuda  _Z14calculateForcePA400_A400_dS1_S1_S1_dddddd    A10  sm_86   

             exeArgs   Block Size      Grid Size model_type  total_AI  \
0  8192 10000 10 100  (

In [23]:
# let's replace the NaN values with -1, since these represent performance counter values that were not gathered.
# df_merged.fillna(-1, inplace=True)

df_merged.to_csv('all-NCU-GPU-Data-with-IMIX.csv', index=False)